# IIT-08 — Parité et broadcast : le couple qui a mordu IIT

**Marche 2 de l'escalier `iit_lean`** (issue #20649, escalier #20414, veine 5 de l'EPIC #16781).
La marche 1 ([IIT-07](https://github.com/jsboige/CoursIA), PR #20419) a établi le vocabulaire de
la coupe et le verdict de décomposabilité : le théorème A dit que l'identité est décomposable.

Ce carnet monte les deux pièces qui manquent à la critique :

- **l'outil parité** (`parity`) et ses quatre lemmes ;
- **le système broadcast-parité** (`broadcast`), dont l'image se réduit à deux états constants ;
- **le théorème B** (`broadcast_integrated`) : le broadcast est *intégré pour toute coupe* —
  c'est « l'objet qui a mordu IIT » érigé en théorème ;
- **le théorème C** (`broadcast_two_cycle`) : la dynamique de ce même objet est *triviale* —
  toute orbite atteint un point fixe en deux pas.

Le couple **B et C** est le contenu formel de la critique d'Aaronson contre le proxy
d'intégration : intégration maximale, comportement nul.

## 0. Ce que ce carnet démontre

1. La parité d'un état se calcule (outil exécutable, mesuré sur des états concrets).
2. Le broadcast uniformise : l'état suivant ne dépend que de la parité de l'état courant.
3. Un oracle par énumération mesure les orbites : point fixe atteint en au plus deux pas,
   et la troisième itération égale toujours la deuxième (théorème C rendu visible).
4. Le théorème B atteste l'intégration universelle du même objet — le contraste avec le
   théorème A de la marche 1 est le cœur du débat.

Le carnet consomme le lake `iit_lean` (kernel `lean4-wsl`) **sans le modifier** : toutes les
définitions et théorèmes viennent de `IIT.Integration`.

In [1]:
import IIT.Integration
open IIT

-- Le contrat du carnet : outil, systeme, theoremes B et C, alias figes
#check @parity
#check @broadcast
#check @broadcast_integrated
#check @broadcast_two_cycle
#check @broadcast_trivial_dynamics

import IIT.Integration
open IIT

-- Le contrat du carnet : outil, systeme, theoremes B et C, alias figes
#check @parity
──────▶  parity : (n : Nat) → State n → Bool
#check @broadcast
──────▶  @broadcast : {n : Nat} → System n
#check @broadcast_integrated
──────▶  @broadcast_integrated : ∀ {n : Nat}, integrated broadcast
#check @broadcast_two_cycle
──────▶  @broadcast_two_cycle : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)
#check @broadcast_trivial_dynamics
──────▶  @broadcast_trivial_dynamics : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)
--% env 0

Raw input:
{"cmd": "import IIT.Integration\nopen IIT\n\n-- Le contrat du carnet : outil, systeme, theoremes B et C, alias figes\n#check @parity\n#check @broadcast\n#check @broadcast_integrated\n#check @broadcast_two_cycle\n#check @broadcast_trivial_dynamics"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data": "parity : (n : Nat) → State n → Bool"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data": "@broadcast : {n : Nat} → System n"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data": "@broadcast_integrated : ∀ {n : Nat}, integrated broadcast"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data":
   "@broadcast_two_cycle : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data":
   "@broadcast_trivial_dynamics : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)"}],
 "env": 0}

Lecture du contrat :

- `parity : (n : Nat) → State n → Bool` — la parité (nombre de bits vrais modulo 2),
  par récurrence sur la taille ;
- `broadcast : System n` — le système : chaque bit de l'état suivant vaut la parité de
  l'état courant ;
- `broadcast_integrated : integrated broadcast` — le théorème B ;
- `broadcast_two_cycle` — le théorème C : `broadcast (broadcast (broadcast s)) = broadcast (broadcast s)` ;
- `broadcast_trivial_dynamics` — l'**énoncé figé n° 3 de #18599**, alias du théorème C sous
  le nom gelé dans l'issue fondatrice du lake.

## 1. L'outil parité

Trois états témoins en taille 3, et des tuples de bits pour *voir* les états (les fonctions
`Fin n → Bool` n'ont pas d'égalité décisionnelle : on compare leurs tuples de bits).

In [2]:
-- Etats temoins en taille 3
def sOneHot : State 3 := fun j => decide (j.val = 0)
def sTrue3  : State 3 := fun _ => true
def sMix3    : State 3 := fun j => decide (j.val = 1)

-- Tuples de bits (l'egalite decisionnelle vit sur les tuples, pas sur les fonctions)
def bits2 (s : State 2) : Bool × Bool := (s 0, s 1)
def bits3 (s : State 3) : Bool × Bool × Bool := (s 0, s 1, s 2)

-- Trois mesures de parite
#eval bits3 sOneHot
#eval bits3 sTrue3
#eval bits3 sMix3
#eval parity 3 sOneHot
#eval parity 3 sTrue3
#eval parity 3 sMix3

-- Etats temoins en taille 3
def sOneHot : State 3 := fun j => decide (j.val = 0)
def sTrue3  : State 3 := fun _ => true
def sMix3    : State 3 := fun j => decide (j.val = 1)

-- Tuples de bits (l'egalite decisionnelle vit sur les tuples, pas sur les fonctions)
def bits2 (s : State 2) : Bool × Bool := (s 0, s 1)
def bits3 (s : State 3) : Bool × Bool × Bool := (s 0, s 1, s 2)

-- Trois mesures de parite
#eval bits3 sOneHot
─────▶  (true, false, false)
#eval bits3 sTrue3
─────▶  (true, true, true)
#eval bits3 sMix3
─────▶  (false, true, false)
#eval parity 3 sOneHot
─────▶  true
#eval parity 3 sTrue3
─────▶  true
#eval parity 3 sMix3
─────▶  true
--% env 1

Raw input:
{"cmd": "-- Etats temoins en taille 3\ndef sOneHot : State 3 := fun j => decide (j.val = 0)\ndef sTrue3  : State 3 := fun _ => true\ndef sMix3    : State 3 := fun j => decide (j.val = 1)\n\n-- Tuples de bits (l'egalite decisionnelle vit sur les tuples, pas sur les fonctions)\ndef bits2 (s : State 2) : Bool \u00d7 Bool := (s 0, s 1)\ndef bits3 (s : State 3) : Bool \u00d7 Bool \u00d7 Bool := (s 0, s 1, s 2)\n\n-- Trois mesures de parite\n#eval bits3 sOneHot\n#eval bits3 sTrue3\n#eval bits3 sMix3\n#eval parity 3 sOneHot\n#eval parity 3 sTrue3\n#eval parity 3 sMix3", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 5},
   "data": "(true, false, false)"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 5},
   "data": "(true, true, true)"},
  {"severity": "info",
   "pos": {"line": 13, "column": 0},
   "endPos": {"line": 13, "column": 5},
   "data": "(false, true, false)"},
  {"severity": "info",
   "pos": {"line": 14, "column": 0},
   "endPos": {"line": 14, "column": 5},
   "data": "true"},
  {"severity": "info",
   "pos": {"line": 15, "column": 0},
   "endPos": {"line": 15, "column": 5},
   "data": "true"},
  {"severity": "info",
   "pos": {"line": 16, "column": 0},
   "endPos": {"line": 16, "column": 5},
   "data": "true"}],
 "env": 1}

Lecture : `(true, false, false)` porte un bit vrai — parité `true` ;
`(true, true, true)` en porte trois — parité `true` ; `(false, true, false)` en porte un —
parité `true`. Les trois états témoins ont une parité impaire : la *valeur* de l'état change,
la parité capture un invariant global (la somme modulo 2 de tous les bits).

In [3]:
-- Les quatre lemmes de l'outil parite
#check @parity_allFalse
#check @parity_allTrue
#check @parity_oneHot_zero
#check @parity_const

-- Les quatre lemmes de l'outil parite
#check @parity_allFalse
──────▶  parity_allFalse : ∀ (n : Nat) (s : State n), (∀ (i : Fin n), s i = false) → parity n s = false
#check @parity_allTrue
──────▶  @parity_allTrue : ∀ {n : Nat}, (parity n fun x => true) = decide (n % 2 = 1)
#check @parity_oneHot_zero
──────▶  parity_oneHot_zero : ∀ (n : Nat), 0 < n → (parity n fun j => decide (↑j = 0)) = true
#check @parity_const
──────▶  parity_const : ∀ (n : Nat) (b : Bool), (parity n fun x => b) = (b && decide (n % 2 = 1))
--% env 2

Raw input:
{"cmd": "-- Les quatre lemmes de l'outil parite\n#check @parity_allFalse\n#check @parity_allTrue\n#check @parity_oneHot_zero\n#check @parity_const", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "parity_allFalse : ∀ (n : Nat) (s : State n), (∀ (i : Fin n), s i = false) → parity n s = false"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@parity_allTrue : ∀ {n : Nat}, (parity n fun x => true) = decide (n % 2 = 1)"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "parity_oneHot_zero : ∀ (n : Nat), 0 < n → (parity n fun j => decide (↑j = 0)) = true"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "parity_const : ∀ (n : Nat) (b : Bool), (parity n fun x => b) = (b && decide (n % 2 = 1))"}],
 "env": 2}

Lecture des quatre lemmes :

- `parity_allFalse` : un état tout-faux a une parité nulle ;
- `parity_allTrue` : un état tout-vrai code **l'imparité de la taille** `n` — la parité d'un
  état uniforme dépend du nombre de bits, pas de leur valeur ;
- `parity_oneHot_zero` : un seul bit vrai (indice 0) rend la parité impaire, quelle que soit
  la taille — c'est le lemme pivot de la preuve du théorème B ;
- `parity_const` : la forme générale — la parité d'un état constant vaut
  `b && decide (n % 2 = 1)`.

In [4]:
-- Mesure du lemme parity_const sur les deux tailles
#eval parity 4 (fun _ => true)
#eval parity 5 (fun _ => true)

-- Mesure du lemme parity_const sur les deux tailles
#eval parity 4 (fun _ => true)
─────▶  false
#eval parity 5 (fun _ => true)
─────▶  true
--% env 3

Raw input:
{"cmd": "-- Mesure du lemme parity_const sur les deux tailles\n#eval parity 4 (fun _ => true)\n#eval parity 5 (fun _ => true)", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "false"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "true"}],
 "env": 3}

Mesure : en taille 4 (pair), l'état tout-vrai a une parité `false` ; en taille 5 (impair),
une parité `true` — exactement `decide (n % 2 = 1)`, le comportement promis par
`parity_const`. Le broadcast en taille paire ne pourra donc jamais stabiliser l'état
tout-vrai : c'est ce que l'oracle confirmera.

## 2. Le système broadcast : un pas

`broadcast s` est l'état **uniforme** dont chaque bit vaut `parity n s` : l'information
globale (la parité) est diffusée partout dans l'état suivant.

In [5]:
-- Un pas de broadcast sur les temoins
#eval bits3 (broadcast sOneHot)
#eval bits3 (broadcast sTrue3)
#eval bits3 (broadcast sMix3)

-- Un pas de broadcast sur les temoins
#eval bits3 (broadcast sOneHot)
─────▶  (true, true, true)
#eval bits3 (broadcast sTrue3)
─────▶  (true, true, true)
#eval bits3 (broadcast sMix3)
─────▶  (true, true, true)
--% env 4

Raw input:
{"cmd": "-- Un pas de broadcast sur les temoins\n#eval bits3 (broadcast sOneHot)\n#eval bits3 (broadcast sTrue3)\n#eval bits3 (broadcast sMix3)", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "(true, true, true)"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 5},
   "data": "(true, true, true)"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 5},
   "data": "(true, true, true)"}],
 "env": 4}

Lecture : les trois états témoins (tous de parité impaire) sont envoyés sur le **même**
état uniforme `(true, true, true)`. Le broadcast oublie tout de l'état courant sauf sa
parité — c'est une projection à une bit d'information, puis une diffusion.

## 3. Oracle : les orbites par énumération

L'oracle énumère les `2^n` états et imprime, pour chaque état de départ, le tuple de bits
après 0, 1, 2 et 3 applications du broadcast.

In [6]:
-- Oracle : enumeration exhaustive des etats par leur code entier
def stateOfNat (n : Nat) (v : Nat) : State n := fun i => ((v >>> i.val) % 2) == 1
def allStates (n : Nat) : List (State n) := (List.range (2 ^ n)).map (stateOfNat n)

#eval (allStates 2).map bits2

-- Oracle : enumeration exhaustive des etats par leur code entier
def stateOfNat (n : Nat) (v : Nat) : State n := fun i => ((v >>> i.val) % 2) == 1
def allStates (n : Nat) : List (State n) := (List.range (2 ^ n)).map (stateOfNat n)

#eval (allStates 2).map bits2
─────▶  [(false, false), (true, false), (false, true), (true, true)]
--% env 5

Raw input:
{"cmd": "-- Oracle : enumeration exhaustive des etats par leur code entier\ndef stateOfNat (n : Nat) (v : Nat) : State n := fun i => ((v >>> i.val) % 2) == 1\ndef allStates (n : Nat) : List (State n) := (List.range (2 ^ n)).map (stateOfNat n)\n\n#eval (allStates 2).map bits2", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "[(false, false), (true, false), (false, true), (true, true)]"}],
 "env": 5}

In [7]:
-- Orbites en taille 2 : (depart, pas 1, pas 2, pas 3)
#eval (allStates 2).map (fun s =>
  (bits2 s,
   bits2 (broadcast s),
   bits2 (broadcast (broadcast s)),
   bits2 (broadcast (broadcast (broadcast s)))))

-- Orbites en taille 2 : (depart, pas 1, pas 2, pas 3)
#eval (allStates 2).map (fun s =>
─────▶  [((false, false), (false, false), (false, false), false, false),
 ((true, false), (true, true), (false, false), false, false),
 ((false, true), (true, true), (false, false), false, false),
 ((true, true), (false, false), (false, false), false, false)]
  (bits2 s,
   bits2 (broadcast s),
   bits2 (broadcast (broadcast s)),
   bits2 (broadcast (broadcast (broadcast s)))))
--% env 6

Raw input:
{"cmd": "-- Orbites en taille 2 : (depart, pas 1, pas 2, pas 3)\n#eval (allStates 2).map (fun s =>\n  (bits2 s,\n   bits2 (broadcast s),\n   bits2 (broadcast (broadcast s)),\n   bits2 (broadcast (broadcast (broadcast s)))))", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data":
   "[((false, false), (false, false), (false, false), false, false),\n ((true, false), (true, true), (false, false), false, false),\n ((false, true), (true, true), (false, false), false, false),\n ((true, true), (false, false), (false, false), false, false)]"}],
 "env": 6}

Lecture de la table (taille 2, paire) :

- chaque ligne converge vers `(false, false)` — l'unique point fixe en taille paire
  (l'état tout-vrai a une parité nulle, cf. la mesure de `parity_const` en taille 4) ;
- `(false, true)` et `(true, false)` passent par `(true, true)` au pas 1 puis atteignent
  leur point fixe au pas 2 : convergence en **au plus deux pas** ;
- **les colonnes 3 et 4 sont identiques partout** : la troisième itération ne change rien —
  le théorème C est déjà visible dans la mesure.

In [8]:
-- Orbites en taille 3 : (depart, pas 1, pas 2, pas 3)
#eval (allStates 3).map (fun s =>
  (bits3 s,
   bits3 (broadcast s),
   bits3 (broadcast (broadcast s)),
   bits3 (broadcast (broadcast (broadcast s)))))

-- Orbites en taille 3 : (depart, pas 1, pas 2, pas 3)
#eval (allStates 3).map (fun s =>
─────▶  [((false, false, false), (false, false, false), (false, false, false), false, false, false),
 ((true, false, false), (true, true, true), (true, true, true), true, true, true),
 ((false, true, false), (true, true, true), (true, true, true), true, true, true),
 ((true, true, false), (false, false, false), (false, false, false), false, false, false),
 ((false, false, true), (true, true, true), (true, true, true), true, true, true),
 ((true, false, true), (false, false, false), (false, false, false), false, false, false),
 ((false, true, true), (false, false, false), (false, false, false), false, false, false),
 ((true, true, true), (true, true, true), (true, true, true), true, true, true)]
  (bits3 s,
   bits3 (broadcast s),
   bits3 (broadcast (broadcast s)),
   bits3 (broadcast (broadcast (broadcast s)))))
--% env 7

Raw input:
{"cmd": "-- Orbites en taille 3 : (depart, pas 1, pas 2, pas 3)\n#eval (allStates 3).map (fun s =>\n  (bits3 s,\n   bits3 (broadcast s),\n   bits3 (broadcast (broadcast s)),\n   bits3 (broadcast (broadcast (broadcast s)))))", "env": 6}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data":
   "[((false, false, false), (false, false, false), (false, false, false), false, false, false),\n ((true, false, false), (true, true, true), (true, true, true), true, true, true),\n ((false, true, false), (true, true, true), (true, true, true), true, true, true),\n ((true, true, false), (false, false, false), (false, false, false), false, false, false),\n ((false, false, true), (true, true, true), (true, true, true), true, true, true),\n ((true, false, true), (false, false, false), (false, false, false), false, false, false),\n ((false, true, true), (false, false, false), (false, false, false), false, false, false),\n ((true, true, true), (true, true, true), (true, true, true), true, true, true)]"}],
 "env": 7}

Lecture de la table (taille 3, impaire) : **deux** points fixes cette fois.

- Les états de parité paire (0 ou 2 bits vrais) convergent vers `(false, false, false)`.
- Les états de parité impaire (1 ou 3 bits vrais) convergent vers `(true, true, true)` —
  qui est lui-même un point fixe : en taille impaire, la parité de l'état tout-vrai vaut
  `true`, l'état se diffuse à lui-même.
- Colonnes 3 = 4 partout, convergence en au plus deux pas : le théorème C tient pour
  les deux tailles.

## 4. Théorème C — la dynamique est triviale

In [9]:
-- Theoreme C et son alias figé
#check @broadcast_two_cycle
#print axioms broadcast_two_cycle
#check @broadcast_trivial_dynamics
#print axioms broadcast_trivial_dynamics

-- Theoreme C et son alias figé
#check @broadcast_two_cycle
──────▶  @broadcast_two_cycle : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)
#print axioms broadcast_two_cycle
──────▶  'IIT.broadcast_two_cycle' depends on axioms: [propext, Quot.sound]
#check @broadcast_trivial_dynamics
──────▶  @broadcast_trivial_dynamics : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)
#print axioms broadcast_trivial_dynamics
──────▶  'IIT.broadcast_trivial_dynamics' depends on axioms: [propext, Quot.sound]
--% env 8

Raw input:
{"cmd": "-- Theoreme C et son alias fig\u00e9\n#check @broadcast_two_cycle\n#print axioms broadcast_two_cycle\n#check @broadcast_trivial_dynamics\n#print axioms broadcast_trivial_dynamics", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@broadcast_two_cycle : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'IIT.broadcast_two_cycle' depends on axioms: [propext, Quot.sound]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "@broadcast_trivial_dynamics : ∀ {n : Nat} (s : State n), broadcast (broadcast (broadcast s)) = broadcast (broadcast s)"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "'IIT.broadcast_trivial_dynamics' depends on axioms: [propext, Quot.sound]"}],
 "env": 8}

L'énoncé : `broadcast (broadcast (broadcast s)) = broadcast (broadcast s)` pour tout état
`s`. Toute orbite atteint un point fixe en deux pas — la **dynamique est triviale** :
l'information globale est préservée (la parité), mais il ne se « passe » plus rien.
L'alias `broadcast_trivial_dynamics` porte le nom figé par l'issue #18599 : le lake garantit
cet énoncé sous ce nom, quelle que soit l'évolution des preuves internes.

In [10]:
-- Temoin concret : le theorique C mesure sur un etat
#eval bits3 (broadcast (broadcast (broadcast sOneHot))) == bits3 (broadcast (broadcast sOneHot))

-- Temoin concret : le theorique C mesure sur un etat
#eval bits3 (broadcast (broadcast (broadcast sOneHot))) == bits3 (broadcast (broadcast sOneHot))
─────▶  true
--% env 9

Raw input:
{"cmd": "-- Temoin concret : le theorique C mesure sur un etat\n#eval bits3 (broadcast (broadcast (broadcast sOneHot))) == bits3 (broadcast (broadcast sOneHot))", "env": 8}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 5},
   "data": "true"}],
 "env": 9}

Le témoin rend `true` : sur l'état témoin, la troisième itération est byte-identique à la
deuxième. La mesure de l'oracle (colonnes 3 = 4) et le théorème C disent la même chose —
l'un pour `2^n` états mesurés, l'autre pour tous les états de toutes les tailles.

## 5. Théorème B — le broadcast est intégré pour toute coupe

In [11]:
-- Theoreme B : l'objet qui a mordu IIT, en theoreme
#check @broadcast_integrated
#print axioms broadcast_integrated
#check @integrated

-- Theoreme B : l'objet qui a mordu IIT, en theoreme
#check @broadcast_integrated
──────▶  @broadcast_integrated : ∀ {n : Nat}, integrated broadcast
#print axioms broadcast_integrated
──────▶  'IIT.broadcast_integrated' depends on axioms: [propext]
#check @integrated
──────▶  @integrated : {n : Nat} → System n → Prop
--% env 10

Raw input:
{"cmd": "-- Theoreme B : l'objet qui a mordu IIT, en theoreme\n#check @broadcast_integrated\n#print axioms broadcast_integrated\n#check @integrated", "env": 9}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "@broadcast_integrated : ∀ {n : Nat}, integrated broadcast"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "'IIT.broadcast_integrated' depends on axioms: [propext]"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data": "@integrated : {n : Nat} → System n → Prop"}],
 "env": 10}

Lecture du théorème B : pour **toute coupe non triviale** (`c.ha` et `c.hb` attestent que
les deux côtés existent), le couple de motifs — côté A diffusé *vrai* **et** côté B diffusé
*faux* — n'est **jamais réalisé ensemble**, alors que chaque motif l'est séparément.

Autrement dit : aucune coupe ne découpe le broadcast en deux moitiés indépendantes. Le
système est **intégré au sens du proxy, universellement** — c'est exactement « l'objet qui a
mordu IIT » (l'exemple qui a fait réaliser à la communauté que le proxy d'intégration
récompensait cet objet), désormais un théorème du lake.

In [12]:
-- Rappel marche 1 : le contraste A vs B
#check @cutIndep_of_sides_independent
#check @product_decomposable

-- Rappel marche 1 : le contraste A vs B
#check @cutIndep_of_sides_independent
──────▶  @cutIndep_of_sides_independent : ∀ {n : Nat} (c : Cut n) (f : System n),
  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →
    (∀ (s s' : State n),
        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →
      cutIndep c f
#check @product_decomposable
──────▶  @product_decomposable : ∀ {n : Nat} (c : Cut n) (f : System n),
  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →
    (∀ (s s' : State n),
        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →
      decomposable f
--% env 11

Raw input:
{"cmd": "-- Rappel marche 1 : le contraste A vs B\n#check @cutIndep_of_sides_independent\n#check @product_decomposable", "env": 10}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@cutIndep_of_sides_independent : ∀ {n : Nat} (c : Cut n) (f : System n),\n  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →\n    (∀ (s s' : State n),\n        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →\n      cutIndep c f"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@product_decomposable : ∀ {n : Nat} (c : Cut n) (f : System n),\n  (∀ (s s' : State n), (∀ (i : Fin n), c.sel i = true → s i = s' i) → ∀ (i : Fin n), c.sel i = true → f s i = f s' i) →\n    (∀ (s s' : State n),\n        (∀ (i : Fin n), c.sel i = false → s i = s' i) → ∀ (i : Fin n), c.sel i = false → f s i = f s' i) →\n      decomposable f"}],
 "env": 11}

Le contraste des deux théorèmes, dans le même vocabulaire :

| | système | verdict |
|---|---|---|
| **Théorème A** (`cutIndep_of_sides_independent`) | l'identité | **décomposable** : chaque moitié vit sa vie |
| **Théorème B** (`broadcast_integrated`) | le broadcast | **intégré pour toute coupe** : aucune moitié ne vit sa vie |

L'identité — le système le plus « riche » en comportement local — est décomposable ; le
broadcast — qui ne fait que diffuser un bit — est universellement intégré. Le proxy inverse
l'intuition.

## 6. La critique d'Aaronson : B et C ensemble

Le couple est la critique :

- **B** : le broadcast est intégré pour toute coupe — information globalement liée ;
- **C** : sa dynamique est triviale — point fixe en deux pas, plus rien ne se passe.

Si un proxy d'intégration attache une valeur maximale à un objet dont le comportement se
réduit à « stabiliser la parité », il ne mesure pas le *calcul* — il mesure un artefact de
la définition. C'est le contenu formel de la critique d'Aaronson contre les théories
de l'information intégrée, et c'est la raison pour laquelle le lake livre les deux
théorèmes **sous le même toit** : l'un sans l'autre ne prouve rien.

In [13]:
-- Les deux theoremes ne dependent que des axiomes standard
#print axioms broadcast_integrated
#print axioms broadcast_two_cycle

-- Les deux theoremes ne dependent que des axiomes standard
#print axioms broadcast_integrated
──────▶  'IIT.broadcast_integrated' depends on axioms: [propext]
#print axioms broadcast_two_cycle
──────▶  'IIT.broadcast_two_cycle' depends on axioms: [propext, Quot.sound]
--% env 12

Raw input:
{"cmd": "-- Les deux theoremes ne dependent que des axiomes standard\n#print axioms broadcast_integrated\n#print axioms broadcast_two_cycle", "env": 11}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "'IIT.broadcast_integrated' depends on axioms: [propext]"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "'IIT.broadcast_two_cycle' depends on axioms: [propext, Quot.sound]"}],
 "env": 12}

Mesure des dépendances : le théorème B ne dépend que de `propext` — même pas de
l'extensionnalité des fonctions ; le théorème C ajoute `Quot.sound` (qui porte `funext`,
utilisée par sa conclusion point à point). Aucune des deux preuves ne repose sur un axiome
ajouté ni sur un `sorry` : le couple B et C est aussi fort que le langage le permet.

## 7. Exercices

Trois exercices, du calcul à la prédiction d'orbite. Les cellules s'exécutent telles
quelles (rien à casser) : compléter le TODO puis re-exécuter.

In [14]:
-- Exercice 1 : la parite a la main
-- TODO etudiant : predire (sans #eval parity) la parite de l'etat ci-dessous,
-- en comptant les bits vrais, puis verifier en decommentant la ligne de controle.
-- Etape 1 : ecrire la valeur attendue (true ou false) en commentaire
-- Etape 2 : decommenter et executer la ligne de controle
def sExo1 : State 3 := fun j => decide (j.val = 2)
#eval "Exercice 1 a completer"
-- #eval parity 3 sExo1

-- Exercice 1 : la parite a la main
-- TODO etudiant : predire (sans #eval parity) la parite de l'etat ci-dessous,
-- en comptant les bits vrais, puis verifier en decommentant la ligne de controle.
-- Etape 1 : ecrire la valeur attendue (true ou false) en commentaire
-- Etape 2 : decommenter et executer la ligne de controle
def sExo1 : State 3 := fun j => decide (j.val = 2)
#eval "Exercice 1 a completer"
─────▶  "Exercice 1 a completer"
-- #eval parity 3 sExo1
--% env 13

Raw input:
{"cmd": "-- Exercice 1 : la parite a la main\n-- TODO etudiant : predire (sans #eval parity) la parite de l'etat ci-dessous,\n-- en comptant les bits vrais, puis verifier en decommentant la ligne de controle.\n-- Etape 1 : ecrire la valeur attendue (true ou false) en commentaire\n-- Etape 2 : decommenter et executer la ligne de controle\ndef sExo1 : State 3 := fun j => decide (j.val = 2)\n#eval \"Exercice 1 a completer\"\n-- #eval parity 3 sExo1", "env": 12}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "\"Exercice 1 a completer\""}],
 "env": 13}

In [15]:
-- Exercice 2 : la taille gouverne le point fixe
-- TODO etudiant : d'apres le lemme parity_const (et la mesure en taille 4 et 5),
-- predire si l'etat tout-vrai de taille 6 est un point fixe du broadcast.
-- Repondre en commentaire : point fixe ou non, et pourquoi (le role de 6 % 2).
#eval "Exercice 2 a completer"


-- Exercice 2 : la taille gouverne le point fixe
-- TODO etudiant : d'apres le lemme parity_const (et la mesure en taille 4 et 5),
-- predire si l'etat tout-vrai de taille 6 est un point fixe du broadcast.
-- Repondre en commentaire : point fixe ou non, et pourquoi (le role de 6 % 2).
#eval "Exercice 2 a completer"
─────▶  "Exercice 2 a completer"

--% env 14

Raw input:
{"cmd": "-- Exercice 2 : la taille gouverne le point fixe\n-- TODO etudiant : d'apres le lemme parity_const (et la mesure en taille 4 et 5),\n-- predire si l'etat tout-vrai de taille 6 est un point fixe du broadcast.\n-- Repondre en commentaire : point fixe ou non, et pourquoi (le role de 6 % 2).\n#eval \"Exercice 2 a completer\"\n", "env": 13}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "\"Exercice 2 a completer\""}],
 "env": 14}

In [16]:
-- Exercice 3 : l'orbite d'un etat donne
-- TODO etudiant : pour n = 2, decrire l'orbite complete de l'etat (true, false) :
-- Etape 1 : donner les etats successifs (pas 1, pas 2, pas 3)
-- Etape 2 : donner le point fixe atteint et le nombre de pas
-- Etape 3 : verifier avec l'oracle (reutiliser la table de la section 3)
#eval "Exercice 3 a completer"


-- Exercice 3 : l'orbite d'un etat donne
-- TODO etudiant : pour n = 2, decrire l'orbite complete de l'etat (true, false) :
-- Etape 1 : donner les etats successifs (pas 1, pas 2, pas 3)
-- Etape 2 : donner le point fixe atteint et le nombre de pas
-- Etape 3 : verifier avec l'oracle (reutiliser la table de la section 3)
#eval "Exercice 3 a completer"
─────▶  "Exercice 3 a completer"

--% env 15

Raw input:
{"cmd": "-- Exercice 3 : l'orbite d'un etat donne\n-- TODO etudiant : pour n = 2, decrire l'orbite complete de l'etat (true, false) :\n-- Etape 1 : donner les etats successifs (pas 1, pas 2, pas 3)\n-- Etape 2 : donner le point fixe atteint et le nombre de pas\n-- Etape 3 : verifier avec l'oracle (reutiliser la table de la section 3)\n#eval \"Exercice 3 a completer\"\n", "env": 14}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 5},
   "data": "\"Exercice 3 a completer\""}],
 "env": 15}

## Conclusion — vers le capstone

La marche 2 est en place : l'outil parité (4 lemmes), le système broadcast, le théorème B
(intégration universelle) et le théorème C (dynamique triviale), tous deux rendus par le
noyau et mesurés par l'oracle d'orbites. La marche 3 (capstone, `IIT-09`) assemblera le
contraste complet : côté Python, le proxy calculé sur des systèmes concrets ; côté Lean,
le couple B et C comme garde-fou formel — ce que le proxy récompense vs ce qu'il
devrait récompenser.